# Geleerde schaalkeuze ×32 of ×64 per MX-blok / A32 — honderd-epochtraining

Deze proef leert **één gedeelde schaalkeuze per MX-blok** voor de twee lagere gewichtsmodi. De actuele E4M3-bron komt uit de FP32-master via de bestaande preprocessing. E2M3 behoudt de hoogste twee exponentbits van die bron. E2M1 ontstaat door twee mantissabits van E2M3 te verwijderen. Beide lagere modi gebruiken dezelfde blokcorrectie: **E8M0-broncode +5 (×32) of +6 (×64)**. De E4M3-bronschaal zelf wordt niet geleerd.

Iedere FP32-hulpwaarde $\theta_b$ start op 6 en blijft in $[5,6]$. De forward kiest $+5$ bij $\theta_b<5{,}5$ en anders $+6$, dus een gelijke waarde kiest ×64. De schaalgradient wordt benaderd als $\ln(2)\sum_{i\in b}g_iq_i$ per lagere modus; de bijdragen van E2M3 en E2M1 worden opgeteld. Na iedere Adam-update volgt projectie op het interval. Er zijn **4.080 extra FP32-hulpwaarden**, met een afzonderlijke Adam-parametergroep en dezelfde beginleersnelheid 0,001. Activaties blijven **A32**; de gedeelde updatemaster en Adam-momenten blijven **FP32**.

Dit notebook leest alleen opgeslagen JSON/CSV. Het start geen training en laadt geen model of checkpoint. De honderd-epochtraining wordt vergeleken met dezelfde gezamenlijk afgewerkte epochs van de bestaande vaste ×32- en ×64-trainingen. Alleen een niet-synthetische, compatibele run krijgt die onderzoeksvergelijking. De notebook is onuitgevoerd meegeleverd; voer zelf alle cellen uit in de resultaatmap.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # De runkopie bevat hier het exacte resultaatpad.
SAVE_FIGURES = False  # True schrijft naar een nieuwe analysis_<tijdstip>-map.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit déze run, ook als een eerdere notebook dezelfde modulenaam gebruikte.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
helper_name = '_mx_triple_learned_scale32_64_reference100_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Epoch 100 en beste checkpoint gekozen door E4M3

Het beste checkpoint is het **eerste strikte maximum van de E4M3-testaccuracy binnen de gemeten training**. Een gelijke score vervangt het eerdere checkpoint niet. E2M3 en E2M1 worden op precies dezelfde epoch gerapporteerd; hun eigen hoogste scores worden niet samengevoegd met een andere E4M3-epoch.

De officiële testset wordt volgens het bestaande protocol voor checkpointselectie gebruikt. De beste score is daardoor geen onafhankelijke testschatting na selectie. Het vaste eindpunt epoch 100 staat er apart naast. Bij een onderbroken run staat de laatste gemeten epoch op de plaats van het eindpunt, met een expliciet onvolledig label. Eén run per methode onderbouwt geen significantieclaim.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Vergelijking met vaste ×32 en vaste ×64

De meegeleverde vaste referenties komen uit de twee bestaande honderd-epochtrainingen. Bij een volledige geleerde run vergelijken we de volledige epochrange. Bij een eerdere stop gebruiken we uitsluitend de gezamenlijk afgewerkte epochs; toekomstige referentiescores tellen dan niet mee in de checkpointselectie. De referentiemetrics en herkomstbestanden worden tegen de meegeleverde hashes gecontroleerd.

De staafgrafieken tonen **epoch 100** en **het beste gezamenlijke checkpoint gekozen op E4M3 binnen de beschikbare epochrange**. Die keuze wordt voor iedere training opnieuw berekend. Alle drie modi komen per training uit dat ene checkpoint; de lagere modi kiezen geen eigen beste epoch. Een gedeeltelijke run toont expliciet de laatste gezamenlijke epoch als onvoltooid resultaat. De curves tonen de hele gemeenschappelijke range, zodat een verschil op één checkpoint in context staat.


In [ ]:
references = ns.load_fixed_references(run)
display(Markdown(ns.comparison_markdown(run, references)))
display(ns.comparison_table(run, references).round(6))
figures['scale_comparison_curves'] = ns.plot_scale_comparison_curves(run, references)
display(figures['scale_comparison_curves'])
plt.close(figures['scale_comparison_curves'])
figures['scale_comparison_checkpoints'] = ns.plot_scale_comparison_checkpoints(run, references)
display(figures['scale_comparison_checkpoints'])
plt.close(figures['scale_comparison_checkpoints'])


## Train- en testaccuracy per modus

De drie modi worden gezamenlijk vanaf een nieuwe initialisatie getraind. Trainaccuracy wordt tijdens de updates verzameld. Testaccuracy wordt na iedere epoch gemeten met de opgeslagen harde blokkeuzes, bevroren evaluatiestaat en volledige testset. De drie modusgebonden BatchNorm-banken leren tijdens de training normaal mee. De E4M3-route blijft direct uit de actuele bron bepaald; E2M3 en E2M1 volgen dezelfde geleerde blokkeuze.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


## Blijft het netwerk verschillende cijfers voorspellen?

De eerdere post-hocproeven lieten bij sommige routes sterk vertekende of constante voorspellingen zien. Daarom wordt hier per testepoch en per modus bijgehouden hoe de voorspellingen over de tien cijfers verdeeld zijn. Een constante voorspelling is direct zichtbaar als **één gebruikte klasse en 100% concentratie**. De tabel toont ook hoe vaak het dominante cijfer werkelijk in de testset voorkomt.

Dit zijn geaggregeerde tellingen; deze notebook doet geen nieuwe inferentie en leidt er geen per-beeldvergelijking uit af.


In [ ]:
display(ns.prediction_table(run, selected_only=True))
figures['prediction_diversity'] = ns.plot_prediction_diversity(run)
display(figures['prediction_diversity'])
plt.close(figures['prediction_diversity'])


## Leren de blokkeuzes werkelijk iets?

De schaaldiagnostiek onderscheidt drie vragen: krijgen de FP32-hulpwaarden een gradient, verplaatsen ze zich, en passeren ze de harde grens van 5,5? Een verplaatsing hoeft nog geen andere forwardkeuze te geven. **Geen harde omschakeling is daarom geen technische FAIL**; het is wel relevant voor de interpretatie van deze training.

De eerste grafiek toont het aandeel ×32 per laag. De histogrammen tonen de volledige θ-verdeling bij initialisatie, bij het E4M3-bestcheckpunt en bij de laatste afgeronde epoch. De helper berekent de harde keuzes en samenvattingen opnieuw uit de opgeslagen blokstaat en vergelijkt deze met CSV en history; er wordt geen model geladen.


In [ ]:
display(Markdown(ns.scale_learning_markdown(run)))
display(ns.scale_table(run).round(8))
figures['scale_choices'] = ns.plot_scale_choices(run)
display(figures['scale_choices'])
plt.close(figures['scale_choices'])


In [ ]:
figures['theta_distributions'] = ns.plot_theta_distributions(run)
display(figures['theta_distributions'])
plt.close(figures['theta_distributions'])


### Wissels, gradients en projectie op de grenzen

Het totaal aantal omschakelingen telt iedere wissel na een optimizerupdate. Een blok kan binnen één epoch meerdere keren wisselen. Het aantal unieke wisselende blokken telt ieder blok maximaal één keer per epoch; het netto verschil vergelijkt alleen de twee epochgrenzen. De L2-update is eveneens een netto verschil en geen opgetelde reisafstand.

De gradientgrafiek toont de grootste waargenomen schaalgradientnorm per laag en per epoch, met een symmetrische logschaal die ook nul kan weergeven. De projectietelling telt hoeveel θ-waarden na Adam buiten het interval vielen en werden teruggezet, inclusief herhaalde projecties van hetzelfde blok. Deze tellingen bewijzen op zichzelf geen accuracywinst.


In [ ]:
figures['scale_switches'] = ns.plot_scale_switches(run)
display(figures['scale_switches'])
plt.close(figures['scale_switches'])


In [ ]:
figures['scale_gradients'] = ns.plot_scale_gradients(run)
display(figures['scale_gradients'])
plt.close(figures['scale_gradients'])


In [ ]:
display(Markdown('## Gemeten trainingsduur en status van de honderd epochs'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Eén opgeslagen E4M3-bron en twee afgeleide exports per geselecteerd checkpoint'))
display(ns.export_table(run))


## Interpretatie en grenzen

De gerichte wijziging tegenover de vaste correcties is dat beide lagere modi hun blokoffset **samen uit +5 of +6 kiezen**, gestuurd door een FP32-hulpwaarde met expliciete benaderde gradient. De E4M3-broncodes en bronschaal blijven volgens dezelfde encoder bepaald. De mantissatruncatie behoudt de E2M3-schaal. De identity-STE naar de mastergewichten blijft behouden, zonder extra factor32/64 in die mastergradient. De afzonderlijke schaalgradient krijgt de som van de twee lagere bijdragen; de detached teacher geeft geen gradient terug naar zijn eigen hogere pass.

Een lage accuracy, constante voorspelling of nul harde wissels blijft een onderzoeksuitkomst wanneer de technische controles slagen. Een dalende soft-target trainingsloss bewijst op zichzelf geen betere classificatie. Beoordeel testaccuracy, harde test-CE, voorspeldistributie en schaalgedrag samen. De proef onderzoekt uitsluitend de keuze tussen ×32 en ×64; zij bewijst niet dat dit de optimale schaalruimte, gradientbenadering of leersnelheid is.

De tijdgrafiek geeft gemeten epochtijden. Bij een volledige run wordt de gemeten som over honderd epochs getoond. Bij een onderbroken run blijft een eventuele projectie nadrukkelijk onderscheiden van metingen. CUDA-uitvoering volgt het bestaande protocol met cuDNN benchmark en heeft geen bitexacte hervatgarantie. De extra FP32-hulpwaarden en Adam-toestand bestaan tijdens training; compacte exports bewaren alleen de harde blokkeuzes. De getoonde GPU-piek is een frameworkmeting en geen bewijs van native MX-kernels of hardwareversnelling.


De gewichten en schaalhulpwaarden gebruiken hetzelfde leersnelheidsschema: 0,001 tijdens epochs 1–50, 0,0001 tijdens 51–75, 0,00001 tijdens 76–90 en 0,000001 tijdens 91–100. De stippellijnen in de epochgrafieken markeren de drie overgangen.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
